In [1]:
!pip install transformers
!pip install sentence-transformers
!pip install pandas
!pip install scikit-learn
!pip install streamlit

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 73.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 78.1 MB/s eta 0:00:00


In [2]:
from transformers import pipeline

classifier = pipeline(
    "sentiment-analysis"
)

[transformers] No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision 714eb0f.
Using a pipeline without specifying a model name and revision in production is not recommended.
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

In [3]:
result = classifier(
    "The app crashes frequently"
)

print(result)

[{'label': 'NEGATIVE', 'score': 0.9996432065963745}]


In [9]:
import pandas as pd

df = pd.read_csv("reviews.csv")

print(df)

                                      review
0  The app crashes every time I upload files
1   Dark mode is missing and hurts usability
2                         Search is too slow
3                 Notifications are annoying
4           I wish there was offline support
5        The app becomes laggy after updates
6               Need voice assistant feature
7              Search results are inaccurate


In [11]:
sentiments = []

for review in df["review"]:

    result = classifier(review)

    sentiments.append(
        result[0]["label"]
    )

df["sentiment"] = sentiments

print(df)

                                      review sentiment
0  The app crashes every time I upload files  NEGATIVE
1   Dark mode is missing and hurts usability  NEGATIVE
2                         Search is too slow  NEGATIVE
3                 Notifications are annoying  NEGATIVE
4           I wish there was offline support  NEGATIVE
5        The app becomes laggy after updates  NEGATIVE
6               Need voice assistant feature  NEGATIVE
7              Search results are inaccurate  NEGATIVE


In [13]:
problems = []

for review in df["review"]:

    review = review.lower()

    if "dark mode" in review:
        problems.append("Dark Mode")

    elif "search" in review:
        problems.append("Search Improvement")

    elif "offline" in review:
        problems.append("Offline Support")

    elif "crash" in review:
        problems.append("Stability Fix")

    else:
        problems.append("Other Issue")

df["problem_detected"] = problems

In [14]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer(
    "all-MiniLM-L6-v2"
)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [18]:
print(df.columns)

Index(['review', 'sentiment', 'problem_detected'], dtype='object')


In [19]:
reviews = df["review"].tolist()

print(reviews)

['The app crashes every time I upload files', 'Dark mode is missing and hurts usability', 'Search is too slow', 'Notifications are annoying', 'I wish there was offline support', 'The app becomes laggy after updates', 'Need voice assistant feature', 'Search results are inaccurate']


In [20]:
embeddings = model.encode(
    reviews)

In [21]:
print(embeddings.shape)

(8, 384)


In [23]:
from sklearn.cluster import KMeans

kmeans = KMeans(
    n_clusters=3
)

clusters = kmeans.fit_predict(
    embeddings
)

df["cluster"] = clusters

In [24]:
feature_map = {

    0: "Improve Search Engine",

    1: "Add Dark Mode",

    2: "Optimize Performance"

}

df["feature_suggestion"] = df[
    "cluster"
].map(feature_map)

In [25]:
feature_data = {

    "Improve Search Engine": {
        "reach": 900,
        "impact": 8,
        "confidence": 90,
        "effort": 6
    },

    "Add Dark Mode": {
        "reach": 700,
        "impact": 6,
        "confidence": 80,
        "effort": 3
    }

}

In [26]:
def calculate_rice(
    reach,
    impact,
    confidence,
    effort
):

    return (
        reach *
        impact *
        confidence
    ) / effort

In [27]:
score = calculate_rice(
    900,
    9,
    85,
    6
)

print(score)

114750.0


In [28]:
priority_scores = {}

In [29]:
for feature in feature_data:

    data = feature_data[feature]

    score = calculate_rice(

        data["reach"],

        data["impact"],

        data["confidence"],

        data["effort"]

    )

    priority_scores[feature] = score

In [30]:
print(priority_scores)

{'Improve Search Engine': 108000.0, 'Add Dark Mode': 112000.0}


In [31]:
sorted_features = sorted(

    priority_scores.items(),

    key=lambda x: x[1],

    reverse=True

)

print(sorted_features)

[('Add Dark Mode', 112000.0), ('Improve Search Engine', 108000.0)]


In [32]:
import pandas as pd

results_df = pd.DataFrame(

    sorted_features,

    columns=[

        "Feature",

        "Priority Score"

    ]

)

print(results_df)

                 Feature  Priority Score
0          Add Dark Mode        112000.0
1  Improve Search Engine        108000.0


In [33]:
!pip install plotly

In [34]:
import plotly.express as px


fig = px.bar(

    results_df,

    x="Feature",

    y="Priority Score",

    title="AI Product Feature Prioritization Dashboard"

)

fig.show()

In [35]:
cluster_count = df["cluster"].value_counts()

print(cluster_count)

cluster
1    3
2    3
0    2
Name: count, dtype: int64


In [36]:
cluster_df = pd.DataFrame(

    cluster_count

).reset_index()

cluster_df.columns = [

    "Cluster",

    "Count"

]

In [37]:
fig = px.pie(

    cluster_df,

    names="Cluster",

    values="Count",

    title="User Complaint Clusters"

)

fig.show()

In [38]:
results_df.to_csv(

    "priority_results.csv",

    index=False

)

print("Exported")

Exported


In [39]:
top_feature = results_df.iloc[0]["Feature"]

print(

    "AI Recommendation: Highest priority feature is",

    top_feature

)

AI Recommendation: Highest priority feature is Add Dark Mode


In [40]:
print("======== PROJECT OUTPUT ========")

print()

print("Total Reviews Analyzed:", len(df))

print()

print("Sentiment Analysis Completed")

print()

print("Embeddings Generated")

print()

print("Complaint Clusters Created")

print()

print("AI Generated Product Features")

print()

print("RICE Prioritization Completed")

print()

print("Top Feature Recommendation:")

print(results_df.iloc[0])

print()

print("Dashboard Generated Successfully")

======== PROJECT OUTPUT ========

Total Reviews Analyzed: 8

Sentiment Analysis Completed

Embeddings Generated

Complaint Clusters Created

AI Generated Product Features

RICE Prioritization Completed

Top Feature Recommendation:
Feature           Add Dark Mode
Priority Score         112000.0
Name: 0, dtype: object

Dashboard Generated Successfully
